# Post-hoc sensitivity analysis

This notebook walks through post-hoc analysis of MADS sensitivity analysis (SA) results using `SA_post_hoc_analysis.py`.

**Run environment:** Execute inside a DVM-DOS-TEM Docker container (`dvmdostem-dev` or `dvmdostem-autocal`). Python dependencies (pandas, matplotlib, bokeh, xarray, netCDF4, scikit-learn, etc.) are pre-installed in those images — do not run `pip install` here. **This notebook is configured to run on GPC VM instance.**

**Workflow data:** SA outputs live under `/data/workflows/` on the container filesystem (the host path mounted as that volume). SA outputs are keyed by `work_dir` in your SA yaml. The bundled Step 1 example [`sa-step1-example-imn.yaml`](../agent_calibration/sa-step1-example-imn.yaml) is ready to run for **CMT04 (shrub tundra) at Imnavait Creek** (`work_dir: /data/workflows/CMT04-IMNAVIAT-sa-N100`). For agent automation, use [`agent_calibration/`](../agent_calibration/) templates in `logs/` instead. For other sites or CMTs, copy the example or [`sa-step1-template.yaml`](../agent_calibration/sa-step1-template.yaml) and adjust the keys listed in the Step 1 section below.

Related docs: [`mads_calibration/README.md`](../README.md), demo notebook [`MADS_calibration_demo.ipynb`](MADS_calibration_demo.ipynb).

**Note:** Older versions utilize ca-config files and MADS calibration with Julia, whereas this notebook works with `mads_calibration/SA_setup_and_run.py`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics as sklm
import scipy.stats

# Resolve mads_calibration on sys.path (Docker: /work/mads_calibration)
MADS_CALIB_DIR = Path('/work/mads_calibration')
if not MADS_CALIB_DIR.is_dir():
    # Running from repo: mads_calibration/notebooks/ -> ../
    _here = Path.cwd().resolve()
    if _here.name == 'notebooks' and (_here.parent / 'SA_post_hoc_analysis.py').exists():
        MADS_CALIB_DIR = _here.parent
    elif (_here / 'mads_calibration' / 'SA_post_hoc_analysis.py').exists():
        MADS_CALIB_DIR = _here / 'mads_calibration'
    else:
        raise RuntimeError(
            'Cannot find mads_calibration; run inside Docker or from the repo root.'
        )
if str(MADS_CALIB_DIR) not in sys.path:
    sys.path.insert(0, str(MADS_CALIB_DIR))

import SA_post_hoc_analysis


### Step 1 SA (cmax → INGPP)

Step 1 explores **cmax** for each active PFT against **GPPAllIgnoringNitrogen** targets (modeled as **INGPP**, GPP without nitrogen limitation). Use [`sa-step1-example-imn.yaml`](../agent_calibration/sa-step1-example-imn.yaml) as-is for **CMT04 at Imnavait Creek**, or copy it to `sa-{SITE}-step1.yaml` (under `logs/` for agent runs) and edit the site-specific keys below.

| Key | Step 1 value | What to change for other sites / CMTs |
| --- | --- | --- |
| `cmtnum` | `4` | Integer CMT number; must match an entry in `calibration/calibration_targets.py`. |
| `site` | `/data/input-catalog/Imnavait` | Path to driving inputs for your location (under `/data/input-catalog/` or elsewhere in the container). |
| `PXx`, `PXy` | `0`, `0` | Grid cell within the driving dataset for your site. |
| `seed_path` | `/work/parameters` | Directory containing parameter files with reasonable starting `cmax` values for your CMT. |
| `observations` | `/work/calibration` | Folder containing `calibration_targets.py` (usually leave as-is in Docker). |
| `params` / `pftnums` | nine `cmax`, PFTs 0–8 | One `cmax` entry per **active** PFT in your CMT. Check `parameters/cmt_bgcvegetation.txt` for which PFT slots are populated (skip placeholder PFTs such as `PFT9`). Lists must be the same length. |
| `percent_diffs` | `0.25` each | Fractional perturbation around each seed `cmax` (e.g. `0.25` = ±25%). Use `p_bounds` instead if you need explicit min/max (not both). |
| `calib_mode` | `GPPAllIgnoringNitrogen` | **Required for Step 1** — turns off DSL and nitrogen feedback during equilibrium. |
| `target_names` | `GPPAllIgnoringNitrogen` | **Required for Step 1** — must match a key in `calibration_targets.py` for your CMT. |
| `work_dir` | `/data/workflows/CMT04-IMNAVIAT-sa-N100` | Unique output directory under `/data/workflows/`; set `WORK_DIR` below to the same path with a trailing `/`. |
| `N_samples` | `100` | Number of Latin hypercube samples (`sampling_method: lhc`). |
| `opt_run_setup` | `--pr-yrs 100 --eq-yrs 200 --sp-yrs 0 --tr-yrs 0 --sc-yrs 0` | Equilibrium-only run for Step 1; spinup/transient/scenario years stay at 0. |

Run the cell below to launch Step 1 SA (`--force` replaces an existing `work_dir`; skip that cell if you are reusing a finished run). Set `WORK_DIR` in the next cell to match `work_dir` from your yaml (trailing `/` required).


In [ ]:
os.chdir(MADS_CALIB_DIR)
# --force clears work_dir before setup so re-runs in this notebook succeed.
!python SA_setup_and_run.py --force agent_calibration/sa-step1-example-imn.yaml  # sa-{SITE}-step1.yaml for your CMT

In [ ]:
# Match work_dir in sa-step1-example-imn.yaml (or your sa-{SITE}-step1.yaml); trailing / required.
WORK_DIR = '/data/workflows/CMT04-IMNAVIAT-sa-N100/'

# Step 2: reassign to your Step 2 SA work_dir before running Step 2 cells.
# WORK_DIR = '/data/workflows/CMT04-IMN-sa-step2/'
# Headless Step 2 (target-first): analyze.py --work-dir WORK_DIR --biome tundra --save-plots --json-out .../step2-result.yaml

In [ ]:
os.chdir(WORK_DIR)
param_props = pd.read_csv('param_props.csv')
sample_matrix = pd.read_csv("sample_matrix.csv")
targets = pd.read_csv('targets.csv', skiprows=1)
results = pd.read_csv('results.csv')

In [ ]:
SA_post_hoc_analysis.plot_spaghetti(results, targets)

### I've included an interactive version of the spaghetti plot
#### This is helpful to zoom into specific targets to isolate ideal samples

In [ ]:
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import HoverTool
import pandas as pd

def plot_spaghetti(results, targets, saveprefix=''):
    """
    Plots one line for each sample (row) in ``results``. Plots targets as dots.
    X axis of plot are for different columns in ``results``. Makes 2 plots, the
    right one uses a log scale for the y axis. The right plot also has a mean line
    (blue).

    Parameters
    ----------
    results : pandas.DataFrame
        One row for each run (sample), one column for each model output variable.

    targets : pandas.DataFrame
        Single row, one column for each target (truth, or observation) value.

    saveprefix : str
        A string that is prepended to the saved filename 'spaghetti_plot.png'

    Returns
    -------
    None
    """
    # Output to notebook
    output_notebook()

    # Create a new plot with a title and axis labels
    p = figure(title="Spaghetti Plot", x_axis_label='Sample Number', y_axis_label='Values', width=800, height=400)

    # Add lines for each sample
    for i, sample in enumerate(results.values):
        sample_folder = f'sample_{i:09d}'
        line = p.line(range(len(sample)), sample, line_color="blue", line_alpha=0.1, legend_label=f"Sample {sample_folder}")
        hover = HoverTool(tooltips=[("Sample", sample_folder)], renderers=[line])
        p.add_tools(hover)

    # Add mean line
    p.line(range(len(results.columns)), results.mean(), line_color="blue", line_width=2, legend_label="Mean")

    # Add targets as dots
    p.circle(range(len(targets.columns)), targets.values[0], color="red", size=8, legend_label="Targets")

    # Show the plot
    show(p)


In [ ]:
plot_spaghetti(results, targets)

## Equilibrium

In [ ]:
total_counts, counts, eq_check, eq_var_check, eq_data, eq_metrics, lim_dict = SA_post_hoc_analysis.equilibrium_check(path=WORK_DIR, slope_lim = 1e-3, eps_lim=1e-5, cv_lim=1)

### Subsets the samples to only those that passed the equilibrium check

In [ ]:
eq_data[eq_data.all(axis=1)]
true_samples = eq_data[eq_data.all(axis=1)].index.tolist()
results2 = results.loc[true_samples]
results2
true_samples = eq_data[eq_data.all(axis=1)].index.tolist()
sample_matrix2 = sample_matrix.loc[true_samples]
sample_matrix2


### Visualize what the subset sample distributions look like

In [ ]:
plot_spaghetti(results2, targets)

### Here I subset the best_params based on the samples that do pass the equilibrium check

In [ ]:
# Valeria's code, for top N runs the output seems to be reversed (ie displays the worst N runs) -Niko
best_params, best_model = SA_post_hoc_analysis.n_top_runs(results2, targets, sample_matrix2, r2lim=None, N=10)
best_params
plot_spaghetti(best_model, targets)

In [ ]:
from IPython.display import display

N_TOP = 10

# n_top_runs sorts ascending by R², so take the last N rows for the best fits
best_params, best_model = SA_post_hoc_analysis.n_top_runs(
    results2, targets, sample_matrix2, r2lim=None, N=len(results2)
)
best_params = best_params.iloc[-N_TOP:]
best_model = best_model.iloc[-N_TOP:]

r2, rmse, mape, re = SA_post_hoc_analysis.calc_metrics(best_model, targets)
summary = pd.DataFrame({
    'R2': r2,
    'RMSE': rmse,
    'MAPE': mape,
}, index=best_params.index)
summary.index.name = 'sample_index'
summary = summary.join(best_params)

plot_spaghetti(best_model, targets)

print(f"{len(true_samples)} samples passed equilibrium; top {N_TOP} by R²:")
display(summary)

best_sample_index = summary.index[-1]
recommended_cmax = best_params.iloc[-1]   # best single sample
# recommended_cmax = best_params.mean()   # alternative: mean of top 10

print(
    f"\nBest sample: {best_sample_index}  "
    f"(R²={summary['R2'].iloc[-1]:.4f}, RMSE={summary['RMSE'].iloc[-1]:.2f})"
)
print("\nRecommended cmax values for Step 2 seed parameters:")
display(recommended_cmax.to_frame('value').T)

print("\nParameter ranges across top 10:")
display(best_params.describe())

SA_post_hoc_analysis.param_and_targets_box_plots(best_params, best_model, targets)

## Step 1 complete — before Step 2

Before running **Step 2 SA**, update the following:

1. **Fix `cmax` in seed parameters** — Write `recommended_cmax` into your `seed_path` files (e.g. a  copy of `/work/parameters`, updating `cmt_calparbgc.txt` for your CMT). Step 2 SA reads initial values from `seed_path`; `cmax` is held fixed and not re-sampled.
2. **Step 2 SA yaml** — New `work_dir`, `params` (e.g. `micbnup`, `kdcrawc`, `kdcsoma`, …), `target_names` (VEGC, NPP, soil pools), and `calib_mode` (typically `VEGC` or `NPPAll`, not `GPPAllIgnoringNitrogen`).
3. **Run Step 2 SA** — `python SA_setup_and_run.py your-step2-config.yaml`
4. **Re-point `WORK_DIR`** below to the Step 2 `work_dir` (with trailing `/`) before running the Step 2 cells.

## Step 2 Vegetation and Soil Targets


### Step 2 SA (integrated veg/soil targets)

Use `logs/sa-IMN-step2.yaml` (agent-created; gitignored) after [`seed_setup.py`](../agent_calibration_step2/seed_setup.py). Copy from [`sa-step2-template.yaml`](../agent_calibration_step2/sa-step2-template.yaml) or see [`sa-step2-example-imn.yaml`](../agent_calibration_step2/sa-step2-example-imn.yaml). Agent workflow: [`agent-instructions-step2.md`](../agent_calibration_step2/agent-instructions-step2.md); handoff: [`step1-transition.md`](../agent_calibration_step2/step1-transition.md).

| Key | Step 2 value (CMT04 Imnavait) | Notes |
| --- | --- | --- |
| `seed_path` | `/data/workflows/CMT04-IMN/parameters-step2` | Fixed Step 1 `cmax`; soil/veg seeds |
| `params` / `pftnums` | 5 soil + 27 `cfall` | `null` for soil; PFT 0–8 for `cfall(0/1/2)` |
| `percent_diffs` | `0.95` each (32) | Refine with `p_bounds` in later iterations |
| `calib_mode` | `VEGC` | N limitation ON |
| `target_names` | `CarbonShallow`, `CarbonDeep`, `CarbonMineralSum`, `AvailableNitrogenSum`, `OrganicNitrogenSum`, `VegCarbon`, `NPPAll`, `EcosystemRespiration` | Keys from `calibration_targets.py` |
| `aux_outputs` | `INGPP y`, `GPP y` | Required for `nitrogen_check` |
| `work_dir` | `/data/workflows/CMT04-IMN-sa-step2` | Reassign `WORK_DIR` below |
| `N_samples` | `5` smoke, then `100` | See `logs/sa-IMN-step2-N100.yaml` |
| `opt_run_setup` | `--eq-yrs 2000` | Longer equilibrium than Step 1 |

Start with params: `[micbnup, kdcrawc, kdcsoma, kdcsompr, kdcsomcr, cfall(0), cfall(1), cfall(2), ...]`


In [ ]:
os.chdir(WORK_DIR)
param_props = pd.read_csv('param_props.csv')
sample_matrix = pd.read_csv("sample_matrix.csv")
targets = pd.read_csv('targets.csv', skiprows=1)
results = pd.read_csv('results.csv')

In [ ]:
SA_post_hoc_analysis.plot_spaghetti(results, targets)

In [ ]:
plot_spaghetti(results, targets)

## See the relationships between targets and specified paramters for belowground and aboveground targets

In [ ]:
SA_post_hoc_analysis.plot_pft_matrix(results,sample_matrix, targets)

In [ ]:
SA_post_hoc_analysis.plot_relationships(results, sample_matrix, targets, parameters =['kdcrawc', 'kdcsoma', 'kdcsompr','kdcsomcr'],  saveprefix='')

### Nitrogen Limitation Check

In [ ]:
# Imnavait / Arctic shrub tundra: use biome='tundra' (not default 'boreal')
n_check, n_counts = SA_post_hoc_analysis.nitrogen_check(WORK_DIR, biome='tundra')

## Target full run checks

In [ ]:
SA_post_hoc_analysis.plot_equilibrium_relationships(path=WORK_DIR,save=False, saveprefix='CMT04-IMNAVIAT')

## Check Equillibrium

In [ ]:
total_counts, counts, eq_check, eq_var_check, eq_data, eq_metrics, lim_dict = SA_post_hoc_analysis.equilibrium_check(path=WORK_DIR, slope_lim = 1e-3, eps_lim=1e-5, cv_lim=1)

In [ ]:
### Target-first ranking (Step 2) — prefer N-passing samples, eq diagnostic only

from IPython.display import display

N_TOP = 10
STEP2_BIOME = 'tundra'  # Imnavait: tundra band INGPP:GPP ~1.4–1.6

n_check, _ = SA_post_hoc_analysis.nitrogen_check(path=WORK_DIR, biome=STEP2_BIOME)
rank_cols = [c for c in targets.columns if not c.startswith('RECO')]
targets_rank = targets[rank_cols]
results_rank = results[rank_cols]

if n_check is not None and n_check['result'].any():
    pool_ids = n_check.index[n_check['result'].astype(bool)].tolist()
    pool_ids = [i for i in pool_ids if i in results_rank.index]
    if pool_ids:
        results_pool = results_rank.loc[pool_ids]
        sm_pool = sample_matrix.loc[pool_ids]
        print(f"{len(pool_ids)} samples pass {STEP2_BIOME} N band; ranking within N-pass pool")
    else:
        results_pool = results_rank
        sm_pool = sample_matrix
        print(f"No N-passing samples in results; ranking full pool (status: target_fit_review)")
else:
    results_pool = results_rank
    sm_pool = sample_matrix
    print(f"No N-passing samples; ranking full pool (status: target_fit_review)")

best_params, best_model = SA_post_hoc_analysis.n_top_runs(
    results_pool, targets_rank, sm_pool, r2lim=None, N=len(results_pool))
best_params = best_params.iloc[-N_TOP:]
best_model = best_model.iloc[-N_TOP:]

r2, rmse, mape, re = SA_post_hoc_analysis.calc_metrics(best_model, targets_rank)
summary = pd.DataFrame({'R2': r2, 'RMSE': rmse, 'MAPE': mape}, index=best_params.index)
summary.index.name = 'sample_index'
if n_check is not None:
    summary = summary.join(n_check[['ratio', 'result']].rename(columns={'result': 'N_pass'}))
summary = summary.join(best_params)

print(f"Eq diagnostic: {eq_data.all(axis=1).sum()}/{len(eq_data)} full pass; "
      f"{( (~eq_data).sum(axis=1) <= 1).sum()} samples with <=1 eq fail")
print(f"Top {N_TOP} by target R² (selected sample must N-pass for apply):")
display(summary)

best_sample_index = summary.index[-1]
recommended_params = best_params.iloc[-1]
selected_n_pass = bool(n_check.loc[best_sample_index, 'result']) if n_check is not None else False

print(f"\nSelected sample: {best_sample_index}  "
      f"(R²={summary['R2'].iloc[-1]:.4f}, RMSE={summary['RMSE'].iloc[-1]:.2f}, N_pass={selected_n_pass})")
display(recommended_params.to_frame('value').T)

SA_post_hoc_analysis.plot_boxplot(results_rank, targets_rank)
SA_post_hoc_analysis.param_and_targets_box_plots(best_params, best_model, targets_rank)

## Additional plots to help visualize above and beloground Model outputs
### These arent necessary but help you zoom into individual samples and further assess stability of model outputs

## Visualize Full EQ run of NPP per PFT

In [ ]:
import os
import xarray as xr
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.layouts import column

output_notebook()  # Enable inline Bokeh plots in Jupyter

def plot_variables_for_samples(variables_to_plot, pft_number=None, targets=None,
                               output_dir="output", sample_folders=None, last_n_years=10):
    if sample_folders is None:
        sample_folders = [f"sample_{i:09d}" for i in range(0, 10)]

    for variable in variables_to_plot:
        non_zero_pfts = []

        # ---- First pass: identify non-zero PFTs ----
        for sample_folder in sample_folders:
            var_path = os.path.join(sample_folder, output_dir, f"{variable}_yearly_eq.nc")
            if os.path.exists(var_path):
                var_data = xr.open_dataset(var_path)

                if 'pft' in var_data.dims:
                    if pft_number is not None:
                        var_data = var_data.where(var_data['pft'] == pft_number, drop=True)
                        if var_data['pft'].size == 0:
                            continue

                    if 'pftpart' in var_data.dims:
                        var_data = var_data.sum(dim='pftpart')

                    pfts = var_data['pft'].values
                    for pft in pfts:
                        pft_data = var_data.sel(pft=pft)
                        if (pft_data[variable].values != 0).any():
                            non_zero_pfts.append(pft)

        non_zero_pfts = list(set(non_zero_pfts))
        plots = []

        # ---- Plotting loop ----
        for pft in non_zero_pfts:
            p = figure(title=f"{variable} - PFT {pft}", x_axis_label="Time", y_axis_label="Value",
                       width=700, height=400, tools="pan,box_zoom,reset,save")

            hover = HoverTool(tooltips=[("Year", "@x"), ("Value", "@y"), ("Sample", "@sample")])
            p.add_tools(hover)

            for sample_folder in sample_folders:
                var_path = os.path.join(sample_folder, output_dir, f"{variable}_yearly_eq.nc")
                if os.path.exists(var_path):
                    var_data = xr.open_dataset(var_path)

                    if 'pft' in var_data.dims:
                        if pft_number is not None:
                            var_data = var_data.where(var_data['pft'] == pft_number, drop=True)
                            if var_data['pft'].size == 0:
                                continue

                        if 'pftpart' in var_data.dims:
                            var_data = var_data.sum(dim='pftpart')

                        pft_data = var_data.sel(pft=pft)
                        pft_data_last_n_years = pft_data.isel(time=slice(-last_n_years, None))

                        time_values = pft_data_last_n_years['time'].values

                        # --- FIX: collapse extra dims so y matches time ---
                        y_data = pft_data_last_n_years[variable]
                        for d in y_data.dims:
                            if d not in ['time']:
                                y_data = y_data.mean(dim=d)
                        y_values = y_data.values  # should be 1D

                        sample_label = [sample_folder] * len(time_values)

                        source = ColumnDataSource(data={'x': time_values, 'y': y_values, 'sample': sample_label})
                        p.line('x', 'y', source=source, line_width=2)

            # ---- Add dashed target line if provided ----
            if targets is not None and pft_number is not None:
                target_col = f"{variable}_pft{pft_number}"  # Adjust to match target column name
                if target_col in targets.columns:
                    target_value = targets[target_col].sum()  # Sum across all samples
                    p.line(x=[time_values.min(), time_values.max()],
                           y=[target_value, target_value], line_dash='dashed', color='red', legend_label='Target')
                else:
                    print(f"Column {target_col} not found in targets.")

            p.legend.click_policy = "hide"
            plots.append(p)

        if plots:
            show(column(*plots))
        else:
            print(f"No data found to plot for {variable}")

# ---- Example Usage ----
variables_to_plot = ['NPP']  # List of variables you want to plot
pft_number =  1             # Specify the PFT number you want to plot
plot_variables_for_samples(variables_to_plot, pft_number, targets=targets, last_n_years=90)


## Vegetation Carbon for each PFT per PFTPART

In [ ]:
import os
import xarray as xr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_variables_for_samples_interactive(
    variables_to_plot,
    pft_number=None,
    targets=None,
    output_dir="output",
    sample_folders=None
):
    if sample_folders is None:
        sample_folders = [f"sample_{i:09d}" for i in range(0, 100)]

    # Labels for PFT parts
    pft_part_labels = ['Leaf', 'Stem', 'Root']

    for variable in variables_to_plot:
        non_zero_pfts = []

        # --- Scan all PFTs with non-zero values ---
        for sample_folder in sample_folders:
            var_path = os.path.join(sample_folder, output_dir, f"{variable}_yearly_eq.nc")
            if os.path.exists(var_path):
                var_data = xr.open_dataset(var_path)
                if 'pft' in var_data.dims:
                    if pft_number is not None:
                        var_data = var_data.where(var_data['pft'] == pft_number, drop=True)
                        if len(var_data['pft']) == 0:
                            continue
                    pfts = var_data['pft'].values
                    for pft in pfts:
                        pft_data = var_data.sel(pft=pft)
                        if (pft_data[variable].values != 0).any():
                            non_zero_pfts.append(pft)

        non_zero_pfts = list(set(non_zero_pfts))

        # --- Plot per PFT ---
        for pft in non_zero_pfts:
            # 1 row x 3 columns (Leaf, Stem, Root)
            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=[f"{variable} - PFT {pft} - {label}" for label in pft_part_labels]
            )

            for i, pft_part in enumerate(range(3)):
                time = None  # reset for each subplot
                for j, sample_folder in enumerate(sample_folders):
                    var_path = os.path.join(sample_folder, output_dir, f"{variable}_yearly_eq.nc")
                    if os.path.exists(var_path):
                        var_data = xr.open_dataset(var_path)
                        if 'pft' in var_data.dims:
                            if pft_number is not None:
                                var_data = var_data.where(var_data['pft'] == pft_number, drop=True)
                                if len(var_data['pft']) == 0:
                                    continue
                            if 'pftpart' in var_data.dims:
                                pft_data = var_data.sel(pft=pft, pftpart=pft_part)
                                time = pft_data['time'].values

                                # --- Flatten extra dimensions beyond time ---
                                y_data = pft_data[variable]
                                for dim in y_data.dims:
                                    if dim not in ['time']:
                                        y_data = y_data.sum(dim=dim)  # sum or mean, you can choose
                                vals = y_data.values

                                fig.add_trace(
                                    go.Scatter(
                                        x=time,
                                        y=vals,
                                        mode='lines',
                                        line=dict(width=1),
                                        name=f"S{j}",  # short legend label
                                        hovertext=[f"{sample_folder}<br>{variable}: {v:.2f}" for v in vals],
                                        hoverinfo="text+x+y"
                                    ),
                                    row=1, col=i+1
                                )

                # --- Add target as red dashed line ---
                if targets is not None and time is not None:
                    target_col = f"{variable}_pft{pft}_{pft_part_labels[i]}"
                    if target_col in targets.columns:
                        target_value = targets[target_col].sum()
                        fig.add_trace(
                            go.Scatter(
                                x=[time[0], time[-1]],
                                y=[target_value, target_value],
                                mode='lines',
                                line=dict(color="red", dash="dash", width=2),
                                name=f"Target {target_col}",
                                hovertext=[f"Target {target_col}: {target_value:.2f}"]*2,
                                hoverinfo="text"
                            ),
                            row=1, col=i+1
                        )

            fig.update_layout(
                height=500, width=1200,
                title_text=f"Interactive Plot for {variable} - PFT {pft}"
            )
            fig.show()


# ---- Example usage ----
variables_to_plot = ["VEGC"]
plot_variables_for_samples_interactive(
    variables_to_plot,
    pft_number=None,
    targets=targets,
    output_dir="output",
    sample_folders=[f"sample_{i:09d}" for i in range(0, 100)]
)


## Soil Targets

In [ ]:
import os
import xarray as xr
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource

output_notebook()

def plot_variables_for_samples_bokeh(variables_to_plot, targets=None, output_dir="output", sample_folders=None, last_years=10):
    if sample_folders is None:
        # Define sample folders if not provided
        sample_folders = [f"sample_{i:09d}" for i in range(0, 50)]

    # Loop through each variable
    for variable in variables_to_plot:
        # Initialize Bokeh figure
        p = figure(title=f"{variable}", x_axis_label='Time', y_axis_label='Value', plot_width=600, plot_height=400)

        # Loop through each sample folder
        for sample_folder in sample_folders:
            # Construct relative path to variable .nc file
            var_path = os.path.join(sample_folder, output_dir, f"{variable}_yearly_eq.nc")

            # Check if file exists
            if os.path.exists(var_path):
                # Read variable data
                var_data = xr.open_dataset(var_path)

                # Slice the data to get only the last 'last_years' years
                var_data = var_data.sel(time=slice(-last_years, None))

                # Create a ColumnDataSource with sample number
                source = ColumnDataSource(data={
                    'time': var_data['time'].values,
                    'value': var_data[variable].values.flatten(),
                    'sample': [sample_folder]*len(var_data['time'].values)
                })

                # Add line and hover tool to the plot
                p.line('time', 'value', source=source, line_width=2)
                hover = HoverTool()
                hover.tooltips = [("Time", "@time"), ("Value", "@value"), ("Sample", "@sample")]
                p.add_tools(hover)

        # Add dashed line for targets if provided
        if targets is not None:
            target_col = f"{variable}"
            if target_col in targets.columns:
                target_value = targets[target_col].sum()  # Sum across all samples
                p.line(x=[var_data['time'].values.min(), var_data['time'].values.max()], y=[target_value, target_value], line_dash='dashed', color='red')

        # Show the plot
        p.legend.click_policy = "hide"
        show(p)

# Define variables to plot
variables_to_plot = ['SHLWC', 'DEEPC','MINEC','AVLN']  # List of variables you want to plot

# Call the function (assuming targets is a pandas DataFrame)
plot_variables_for_samples_bokeh(variables_to_plot, targets=targets, last_years=60)

